# LLM Zoomcamp 1.15 — Frameworks / ToyAIKit

根据[课程原文](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/15-frameworks.md)整理的实验 notebook。将上一节手写的 agent loop 替换为 ToyAIKit，并保留独立运行所需的准备代码。

在 VS Code 右上角选择本项目的 Python kernel，然后从上往下运行。FAQ 加载需要网络；提问相关 cell 会调用 OpenAI API 并产生费用。输出初始为空，便于记录自己的实验。

## 1. 安装 ToyAIKit

运行下面的 cell 将 ToyAIKit 安装到当前 notebook kernel。安装后若导入失败，重启 kernel 再继续。已有的项目环境包含其余课程依赖。

课程使用终端命令 `uv add toyaikit`；如果希望同时更新项目依赖和锁文件，可在项目根目录执行该命令，替代下面的安装 cell。

In [ ]:
%pip install toyaikit

## 2. 导入并读取环境变量

项目根目录的 `.env` 应包含 `OPENAI_API_KEY`。这里只检查是否存在，不显示密钥。`MODEL` 可以按需修改。

In [1]:
import os
from dotenv import load_dotenv
from ingest import load_faq_data, build_index

from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("请先在项目根目录的 .env 中设置 OPENAI_API_KEY，然后重新运行本 cell。")

MODEL = "gpt-5.4-mini"

## 3. 加载 FAQ 并建立索引

复用项目的 `ingest.py`，无需先运行其他 notebook。这一步从课程 FAQ 网站获取数据。

In [2]:
documents = load_faq_data()
index = build_index(documents)
print(f"Loaded {len(documents)} FAQ documents.")

Loaded 1470 FAQ documents.


## 4. 手动注册工具

先沿用上一节的搜索函数和手写 schema：Python 函数执行搜索，schema 向模型描述工具。

In [3]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [4]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [5]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'Search query text to look up in the course FAQ.'}},
   'required': ['query'],
   'additionalProperties': False}}]

## 5. 自动生成 schema

给函数添加类型提示和 docstring 后，可以省略手写 schema。下面重新定义 `search`，再使用新的 `Tools` 容器注册它，供后续 runner 使用。

这里将返回类型写为 `list[dict]`，对应索引实际返回的文档列表；课程示例的 `dict[str, str]` 与该返回结构不一致。

In [6]:
def search(query: str) -> list[dict]:
    """Search the FAQ database for entries matching the given query."""
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"},
    )

In [7]:
agent_tools = Tools()
agent_tools.add_tool(search)
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

## 6. 配置提示词和 runner

沿用上一节的规则：多次搜索，并根据结果扩展关键词。runner 封装了 API 请求、工具调用、结果回传和循环；callback 展示运行过程。修改规则或工具后，请重新运行 runner 的创建 cell。

In [8]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [9]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model=MODEL),
)

## 7. 单次提问

故意保留课程里的拼写 `Olama`，观察模型是否尝试修正为 `Ollama` 并重新搜索。具体行为由模型决定，每次结果可能不同。也可以改为上一节的报名问题。

In [10]:
question = "How do I run Olama locally?"
# question = "I just discovered the course. Can I join it?"

result = runner.loop(
    prompt=question,
    callback=callback,
)

-> Response received


-> Response received


-> Response received


## 8. 查看费用、token 信息与消息历史

`result` 是框架的结果对象。费用来自框架按 token 用量的估算，实际账单以服务端为准。直接显示 `result` 可检查当前版本提供的 token 字段。

In [11]:
result.cost

CostInfo(input_cost=Decimal('0.00404175'), output_cost=Decimal('0.001782'), total_cost=Decimal('0.00582375'))

In [12]:
result

LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"Olama local run install Ollama locally FAQ"}', call_id='call_EvGwAmBPEB1VS8Yc70NCTcXc', name='search', type='function_call', id='fc_06fb4463ee689595006afa54869bac87d18b72ca83012c9d45', async_=None, caller=None, namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_EvG

In [13]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama local run install Ollama locally FAQ"}', call_id='call_EvGwAmBPEB1VS8Yc70NCTcXc', name='search', type='function_call', id='fc_06fb4463ee689595006afa54869bac87d18b72ca83012c9d45', async_=None, caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_EvGwAmBPEB1VS8Yc70NCTc

## 9. 携带历史继续对话

将上一轮的完整历史传入 `previous_messages`，模型才能知道“另一个模型”指的是之前讨论的 Ollama。若上面改了问题，也请相应修改这里的追问。

In [14]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


## 10. 交互聊天（可选）

单独运行下面的 cell，按输入提示连续提问，输入 `stop` 退出。如果当前界面不支持交互输入，可继续使用上一节的 `runner.loop(...)` 方式。

In [ ]:
runner.run()

-> Response received


-> Response received


-> Response received


## 可以尝试的实验

- 修改 `question`，观察搜索关键词与调用次数。
- 修改 `instructions`，比较多次搜索与较少搜索的表现；修改后重新创建 runner。
- 分别带上和省略 `previous_messages`，比较追问时的上下文理解。
- 比较手动 schema 和自动生成 schema。

ToyAIKit 是本课使用的教学与实验框架。原有手写循环可在 [agentic loop.ipynb](./agentic%20loop.ipynb) 中对照阅读。